In [9]:
import pickle
import pdfplumber
import docx
import re
from collections import Counter

# Define category weights
WEIGHTS = {
    "skills": 0.3,
    "experience": 0.3,
    "education": 0.2,
    "projects": 0.1,
    "certifications": 0.1
}

# Extracted word categories
SKILL_KEYWORDS = ["python", "machine learning", "nlp", "flask", "tensorflow", "django", "sql", "data science"]
EXPERIENCE_KEYWORDS = ["experience", "worked", "intern", "developer", "software engineer", "business analyst"]
EDUCATION_KEYWORDS = ["bachelor", "master", "phd", "degree", "university", "college"]
PROJECTS_KEYWORDS = ["project", "developed", "built", "created", "implemented"]
CERTIFICATIONS_KEYWORDS = ["certification", "certified", "course", "completed"]

class ResumeScorer:
    def __init__(self):
        self.weights = WEIGHTS
        self.skill_keywords = SKILL_KEYWORDS
        self.experience_keywords = EXPERIENCE_KEYWORDS
        self.education_keywords = EDUCATION_KEYWORDS
        self.projects_keywords = PROJECTS_KEYWORDS
        self.certifications_keywords = CERTIFICATIONS_KEYWORDS

    def extract_text(self, file_path):
        text = ""
        if file_path.endswith(".pdf"):
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() or ""
        elif file_path.endswith(".docx"):
            doc = docx.Document(file_path)
            for para in doc.paragraphs:
                text += para.text + " "
        return text.lower().strip() if text else "no_text_found"

    def extract_words(self, text):
        """Extract words and count their frequency."""
        words = re.findall(r'\b[a-zA-Z]+\b', text)  # Extract words only
        return Counter(words)  # Return word frequency dictionary

    def score_resume(self, text):
        if text == "no_text_found":
            return {"score": 0, "word_frequencies": {}}

        word_counts = self.extract_words(text)

        # Count relevant words in each category
        category_scores = {
            "skills": sum(word_counts[word] for word in self.skill_keywords if word in word_counts),
            "experience": sum(word_counts[word] for word in self.experience_keywords if word in word_counts),
            "education": sum(word_counts[word] for word in self.education_keywords if word in word_counts),
            "projects": sum(word_counts[word] for word in self.projects_keywords if word in word_counts),
            "certifications": sum(word_counts[word] for word in self.certifications_keywords if word in word_counts)
        }

        # Compute total weighted score
        total_score = sum(category_scores[cat] * self.weights[cat] for cat in category_scores)
        return {"score": round(total_score * 10, 2), "word_frequencies": dict(word_counts)}

# Save the model
scorer = ResumeScorer()
with open("model.pkl", "wb") as model_file:
    pickle.dump(scorer, model_file)

print("✅ Model saved as model.pkl")


✅ Model saved as model.pkl
